# Semantic context paging

**Recipe 46** · Level 4 (Advanced) · Decision type: `Score`

Judge stored context chunks for relevance to the current task so Python can retain, page out, or rehydrate them while preserving original content and source links.

Sources:

- S01: [TypeSafe AI: Introduction](https://docs.typesafe.ai/introduction)
- S02: [TypeSafe AI: Primitives (Questions)](https://docs.typesafe.ai/primitives)
- S03: [TypeSafe AI: Confidence](https://docs.typesafe.ai/confidence)
- S04: [TypeSafe AI: How to build with TypeSafe](https://docs.typesafe.ai/concepts/how-to-build-with-system-one)

## What you will build

A context store keeps small chunks of prior material -- ticket messages, runbook steps, code comments, monitoring alerts -- that an on-call assistant might need for the task in front of it right now. Only a few chunks can stay "hot" in context at once, so something has to decide which ones. Jev rates every stored chunk's relevance to the current task on a four-level [`Score`](../../docs/glossary.md#score) rubric; Python turns that judgment, plus a fixed budget, into a retain / page-out / rehydrate decision and never loses a chunk's content or source link while doing it. By the end you will have seen one rated chunk up close, the rule that trusts or withholds a rating, the budget that turns trusted ratings into paging decisions, a baseline this recipe is honestly compared against (keep the most recent chunks), and a three-turn session where a chunk is paged out and later rehydrated with its content intact.


## Setup and run mode

The notebook runs from its own folder with the standard library and `jev_cookbook` only. `get_backend(fixtures=...)` replays stored answers offline; `JEV_COOKBOOK_LIVE=1` (see the README) swaps in live calls and changes nothing else. The header states which mode ran, and every number below carries that mode with it. The fixture set has 69 scored-or-demo rows: 30 validation chunk ratings across 6 tasks, 35 test chunk ratings across 7 task-level units (4 independent tasks plus a 3-turn session), and 4 demo chunks shown individually below.


In [ ]:
from jev_cookbook import get_backend, load_helpers
from jev_cookbook.evaluation import (
    confusion_matrix,
    evaluate_outcomes,
    evaluate_selective,
    exact_agreement,
    mean_absolute_error,
    mean_recall_at_budget,
    recall_at_budget,
    score_level,
    select_confidence_threshold,
    selective_curve,
)
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path
from jev_cookbook.style import (
    apply_style,
    plot_confusion_matrix,
    plot_risk_coverage,
    run_header,
    show_answer,
)

apply_style()
helpers = load_helpers()  # this recipe's helpers.py, loaded by path
examples = load_inputs()
labels = load_labels()
backend = get_backend(fixtures=responses_path())
questions = helpers.build_questions()
# N for the header is the number of chunk ratings the metrics are about: every example with a
# gold label, which excludes the 4 demo chunks.
scored = [e for e in examples if e.split not in ("train", "demo")]

offline = backend.mode in ("synthetic", "scripted")
check = " (a pipeline check, not a Jev result)" if offline else ""
# CONTRIBUTING.md section 2: "a number from the validation split is a selection step, not a
# result." This label does not depend on the run mode, unlike `check` above.
selection = " (a selection step, not a reported result)"

# Every one of the 69 rows in fixtures/ needs at most one request, and no two rows ever ask the
# same (task, chunk) pair, so this cache is never actually needed to avoid a duplicate call --
# it is here because a reader who copies this notebook's pattern into a recipe that DOES repeat
# a request should inherit the cache, not the bug.
answered = {}


def decide(example):
    if example.id not in answered:
        answered[example.id] = backend.decide(helpers.build_state(example.fields), questions)[
            "relevance"
        ]
    return answered[example.id]


run_header(
    46,
    "Semantic context paging",
    backend=backend,
    n_examples=len(scored),
)
print(
    f"fixtures: {len(examples)} rows total "
    f"({len(scored)} scored, {len(examples) - len(scored)} demo){check}"
)

## The state

Jev sees exactly two things: the task description and the chunk's own text. `chunk_id`, `source` (the chunk's source reference, such as `ticket:5521#msg2` or `runbook:refund-policy-v3#step4`) and `recency_rank` (how many chunks have been added to this task's pool more recently than this one -- 0 is the newest) all stay in Python. None of the three reaches the model; all three are reattached to the outcome afterwards, which is what lets every paging decision trace back to its source and lets the recency baseline this recipe compares against be computed at all -- Jev is never asked how old a chunk is.


In [ ]:
demo = {e.fields["chunk_id"]: e for e in examples if e.split == "demo"}
example = demo["needed"]
print("Python has:")
print(example.fields)
state = helpers.build_state(example.fields)
print("Jev sees:")
print(state)

## The questions

There is one question per chunk, and it asks one thing: how relevant is this chunk to the current task? [`Score`](../../docs/glossary.md#score) fits because relevance here is a matter of degree, not a label: a chunk can be squarely on topic without stating what the task actually needs. Every level is described in words, lowest first, and judged on its own -- Jev never sees a level's number or its neighbours, so each description stands alone rather than saying "more relevant than the level above". The instructions tell Jev explicitly not to use recency as a signal, because recency is the separate, zero-request baseline Python compares against later, not something the rubric should reward on its own.


In [ ]:
relevance = questions["relevance"]
print(relevance.instructions)
for level, description in enumerate(relevance.criteria):
    print(f"  level {level}: {description}")

## One answer up close

Before any aggregate, look at individual typed answers. A `Score` answer holds the expected level (`score`, the probability-weighted average -- a float, since the rubric levels are ordered), a probability per level, and a [`confidence`](../../docs/glossary.md#confidence) derived from how concentrated those probabilities are around the top level, not from the top probability alone. All four chunks below are rated against the same task -- "is it safe to deploy the invoicing service right now" -- so the only thing that varies between them is what each chunk itself states.


In [ ]:
not_relevant_example = demo["not-relevant"]
not_relevant_answer = decide(not_relevant_example)
show_answer(not_relevant_answer)

A banner-copy request from the marketing team has nothing to do with whether it is safe to deploy the invoicing service: the rated level is 0, `NOT_RELEVANT`, with high confidence.


In [ ]:
supporting_example = demo["supporting"]
supporting_answer = decide(supporting_example)
show_answer(supporting_answer)

A clean deploy history three days ago is genuinely useful context -- it rules out "this service is already unstable" -- but it does not, on its own, say whether a freeze is in effect right now. That is `SUPPORTING` (level 2), and its confidence is lower: there is a real case for reading it as merely background.


In [ ]:
needed_answer = decide(example)  # the chunk shown in "The state" above
show_answer(needed_answer)

The deploy-freeze window is the one chunk that, on its own, answers the task: `NEEDED` (level 3), with high confidence.


In [ ]:
adversarial_example = demo["adversarial"]
adversarial_answer = decide(adversarial_example)
show_answer(adversarial_answer)

This chunk is the lexical-overlap trap: it contains the word "freeze" twice, by name, in a pipeline that happens to be called `invoicing-deploy-freeze-check`. A purely keyword-driven judge would score this high. It states nothing about a freeze actually being in effect, so its gold level is `NOT_RELEVANT` -- and this demo chunk carries no label (demo examples are never scored), but two scored chunks elsewhere in the fixtures carry exactly this trap, rated confidently wrong on purpose, so the evaluation below has a real instance of it to find rather than only this illustration.


## Python's part

Jev supplies the relevance judgment; everything downstream of it is code. `helpers.classify_chunk` decides whether to trust a chunk's rating at all: below a [confidence gate](../../docs/glossary.md#confidence-gate), the outcome is an explicit [`review`](../../docs/glossary.md#review), whatever level the answer names, because a spread-out distribution over four levels is not solid ground for a paging decision. At or above the gate, a fixed retention cutoff applies: a trusted chunk whose most likely level falls below `SUPPORTING` can never be retained, however much budget is free, so it is `paged_out` directly. Everything else becomes a `candidate` -- eligible for a budget slot, not yet promised one; `helpers.allocate_budget` (next section) is the separate mechanism that spends the budget. Nothing the model says can retain more chunks than the budget allows or invent a fifth outcome, because both rules live in `helpers.py` and are tested against a grid of synthesized answers across every rubric level and confidence, not only the chunks below.

The confidence gate itself is chosen here, on `validation`, and frozen for the rest of this notebook. `select_confidence_threshold(..., min_coverage=0.85)` is used rather than `target_accuracy=1.0`: it asks for the best accuracy among thresholds that still rate at least 85% of validation chunks, rather than forcing perfect validation accuracy (which this pool does not have -- two validation chunks are genuinely ambiguous, rated wrong at low confidence on purpose, and `target_accuracy=1.0` would have to review past them both to claim it).


In [ ]:
val_examples = [e for e in examples if e.split == "validation"]
val_answers = [decide(e) for e in val_examples]
val_gold = [labels[e.id] for e in val_examples]
val_correct = [score_level(a) == g for a, g in zip(val_answers, val_gold, strict=True)]
val_confidence = [a.confidence for a in val_answers]
gate = select_confidence_threshold(val_correct, val_confidence, min_coverage=0.85)
print(f"chosen confidence gate, frozen from here on: {gate:.4f}{selection}{check}")

for name, _chunk_example, chunk_answer in (
    ("not-relevant", not_relevant_example, not_relevant_answer),
    ("supporting", supporting_example, supporting_answer),
    ("needed", example, needed_answer),
):
    result = helpers.classify_chunk(name, chunk_answer, gate)
    print(
        f"{name}: level {result.level} at confidence {chunk_answer.confidence:.2f} "
        f"-> {result.outcome} ({result.reason})"
    )

## Evaluation: recall of needed chunks under the budget

Every task's chunk pool is scored, then `helpers.allocate_budget` spends a fixed budget of `helpers.BUDGET_SLOTS` (3) retained slots among the chunks `classify_chunk` called `candidate`, highest expected score first. The use case names a baseline to compare against: **keep the most recently added chunks**, which costs zero requests and is exactly what a system without semantic judgment would do. `jev_cookbook.evaluation.recall_at_budget` answers the same question for both: of the chunks a task's own gold label marks as `NEEDED` (level 3, the ones the task's answer or action actually depends on), what share would a budget of 3 chunks find, ranked by this score?

Whether that is a fair comparison depends on how the chunk pools are built. If every task's `NEEDED` chunk were deliberately the oldest one, the recency baseline would lose by construction and the margin below would measure the fixtures, not the mechanism. So the pools below are built both ways: in half of them the `NEEDED` chunk is the most recently added one (the recency baseline finds it for free, same as the semantic score should); in the other half it is the oldest chunk in the pool, with newer irrelevant chunks added after it. The table marks which is which, so the margin can be read task by task rather than taken on faith.


In [ ]:
# One task, concrete: v2 is recency-wrong (the NEEDED chunk, r4, is the oldest of five).
v2_examples = [e for e in val_examples if e.fields["task_id"] == "v2"]
v2_examples = sorted(v2_examples, key=lambda e: e.fields["recency_rank"])
print(f"task: {v2_examples[0].fields['task']}\n")
print("by recency (rank 0 = newest; the recency baseline keeps the first 3 rows):")
for e in v2_examples:
    gold = labels[e.id]
    print(
        f"  rank {e.fields['recency_rank']}  {e.fields['chunk_id']}  gold level {gold}  "
        f"{e.fields['chunk'][:58]}..."
    )

v2_scores = {e.fields["chunk_id"]: decide(e).score for e in v2_examples}
print("\nby Jev's expected score (the policy keeps the top 3):")
for e in sorted(v2_examples, key=lambda e: -v2_scores[e.fields["chunk_id"]]):
    cid = e.fields["chunk_id"]
    print(
        f"  score {v2_scores[cid]:.2f}  rank {e.fields['recency_rank']}  {cid}  "
        f"gold level {labels[e.id]}"
    )

v2_classifications = [
    helpers.classify_chunk(e.fields["chunk_id"], decide(e), gate) for e in v2_examples
]
v2_decisions, v2_budget = helpers.allocate_budget(v2_classifications, v2_scores, {}, slots=3)
print(f"\nbudget used: {v2_budget.used('slots')} of {v2_budget.limit('slots')}")
for d in v2_decisions:
    print(f"  {d.chunk_id}: {d.classification.outcome} -> {d.status}")

In [ ]:
def task_ids(split_examples):
    seen = []
    for e in split_examples:
        if e.fields["task_id"] not in seen:
            seen.append(e.fields["task_id"])
    return seen


def recency_right(task_examples):
    needed_ranks = [e.fields["recency_rank"] for e in task_examples if labels[e.id] == 3]
    return bool(needed_ranks) and min(needed_ranks) == 0


def recall_table(split_examples):
    rows = []
    for task_id in task_ids(split_examples):
        task_examples = [e for e in split_examples if e.fields["task_id"] == task_id]
        gold_relevant = [labels[e.id] == 3 for e in task_examples]
        policy_scores = [decide(e).score for e in task_examples]
        baseline_scores = [-e.fields["recency_rank"] for e in task_examples]
        rows.append(
            (
                task_id,
                recall_at_budget(gold_relevant, policy_scores, helpers.BUDGET_SLOTS),
                recall_at_budget(gold_relevant, baseline_scores, helpers.BUDGET_SLOTS),
                "recency-right" if recency_right(task_examples) else "recency-wrong",
            )
        )
    return rows


for split_name, split_examples, label in (
    ("validation", val_examples, selection + check),
    ("test", [e for e in examples if e.split == "test"], check),
):
    rows = recall_table(split_examples)
    print(f"{split_name}, {len(task_ids(split_examples))} task-level units{label}")
    for task_id, policy_recall, baseline_recall, shape in rows:
        print(
            f"  {task_id:6s} ({shape:13s}) policy recall {policy_recall:.4f}  "
            f"baseline recall {baseline_recall:.4f}{label}"
        )
    queries_policy = [
        (
            [labels[e.id] == 3 for e in split_examples if e.fields["task_id"] == t],
            [decide(e).score for e in split_examples if e.fields["task_id"] == t],
        )
        for t in task_ids(split_examples)
    ]
    queries_baseline = [
        (
            [labels[e.id] == 3 for e in split_examples if e.fields["task_id"] == t],
            [-e.fields["recency_rank"] for e in split_examples if e.fields["task_id"] == t],
        )
        for t in task_ids(split_examples)
    ]
    mean_policy = mean_recall_at_budget(queries_policy, helpers.BUDGET_SLOTS)
    mean_baseline = mean_recall_at_budget(queries_baseline, helpers.BUDGET_SLOTS)
    n_requests = len(split_examples)
    print(
        f"  mean recall@{helpers.BUDGET_SLOTS}: policy {mean_policy:.4f} ({n_requests} "
        f"requests)  baseline {mean_baseline:.4f} (0 requests){label}"
    )

The policy reaches a mean recall of 1.0000 on both splits: every task's `NEEDED` chunk lands in the top 3 by Jev's expected score. The recency baseline matches it exactly on the `recency-right` tasks (3 of 6 on validation, 3 of 7 on test, including one turn of the session) and scores 0.0000 on every `recency-wrong` task, for a mean of 0.5000 on validation and 0.4286 on test -- a real, task-by-task margin, not a single aggregate number asserted on faith.

**What would have to change in the data for the recency baseline to win?** On the `recency-right` tasks, nothing could make it lose: the baseline and the policy are both looking at the same correct answer, and a tie favours whichever cost less (the baseline, at zero requests). On the `recency-wrong` tasks, the baseline cannot win as this pool is built, either -- its `NEEDED` chunk is the oldest one by construction. What *would* change the headline is the **mix**: a pool built entirely of `recency-right` tasks would make the two methods tie overall at zero marginal benefit from semantic judgment, which is why this notebook does not build one. The margin above is earned by the `recency-wrong` tasks existing at all, and the table names exactly which tasks they are.

This is also a loss the policy is not immune to: the policy's mean recall is 1.0000 here because every `NEEDED` chunk in these fixtures happens to be rated correctly at high confidence (see "One answer up close" and the rubric evaluation below for a case where Jev's own rating is wrong) -- not because `recall_at_budget` or `allocate_budget` could not report a lower number. A wrong, confident rating that outranked a task's one `NEEDED` chunk would show up here as a recall below 1.0000, exactly as it should.


## Rubric-level agreement

Separately from the budget-level recall above, how often does Jev's most likely level (`score_level`, ties going to the lower level) exactly match the [gold label](../../docs/glossary.md#gold-label)? This is a Score-calibration check, not a paging decision: it is the number that shows where the fixtures' one confidently-wrong, lexical-overlap chunk per split actually lands.


In [ ]:
test_examples = [e for e in examples if e.split == "test"]
test_answers = [decide(e) for e in test_examples]
test_gold = [labels[e.id] for e in test_examples]

for split_name, split_examples, split_answers, split_gold, label in (
    ("validation", val_examples, val_answers, val_gold, selection + check),
    ("test", test_examples, test_answers, test_gold, check),
):
    print(f"{split_name}, {len(split_examples)} chunk ratings{label}")
    print(f"  exact-level agreement: {exact_agreement(split_gold, split_answers):.4f}{label}")
    print(
        f"  mean absolute error: {mean_absolute_error(split_gold, split_answers):.4f} levels{label}"
    )

predicted_levels = [score_level(a) for a in test_answers]
matrix = confusion_matrix(test_gold, predicted_levels, labels=[0, 1, 2, 3])
width = max(len(str(label)) for label in matrix.labels)
leader_width = max(width + len("gold "), len("predicted ->"))
header = "  ".join(f"{label:>{width}d}" for label in matrix.labels)
print(f"\nTest confusion matrix, gold level rows by predicted level columns{check}:")
print(f"  {'predicted ->':<{leader_width}s}  {header}{check}")
for gold_label, row in zip(matrix.labels, matrix.matrix.tolist(), strict=True):
    counts = "  ".join(f"{count:{width}d}" for count in row)
    leader = f"gold {gold_label:<{width}d}"
    print(f"  {leader:<{leader_width}s}  {counts}{check}")
plot_confusion_matrix(matrix, title=f"Test split, rubric levels{check}")

Row 1 (gold `BACKGROUND`), column 3 (predicted `NEEDED`) holds the one test-split mistake: a chunk about SSO redirect error logging, rated `NEEDED` at high confidence because its own words -- "SSO redirect errors" and "error code" -- echo the task's "redirects to an error page" almost exactly. It is the test-split twin of the "freeze" chunk shown earlier, and unlike that purely illustrative demo chunk, this one is scored: it is what makes the test exact-level agreement 0.9714 rather than a perfect 1.0000, and it is the confidently-wrong answer `tests/test_helpers.py::test_stored_answers_are_not_all_right` requires.


## Trusting a rated level: risk and coverage

A second, narrower question than "is the retention decision right" (next section): if Jev's rated level is trusted whenever its confidence clears some gate, how does accuracy trade off against how much gets trusted? This is a confidence-only selective question -- no retention cutoff, no budget -- so `evaluate_selective` and `selective_curve` apply directly.


In [ ]:
selective = evaluate_selective(
    [score_level(a) == g for a, g in zip(test_answers, test_gold, strict=True)],
    [a.confidence for a in test_answers],
    gate,
)
print(
    f"answered: {selective.n_answered} of {selective.n_total} "
    f"(coverage {selective.coverage:.4f}){check}"
)
print(f"accuracy among those answered: {selective.accuracy:.4f}{check}")
print(f"risk among those answered: {selective.risk:.4f}{check}")

curve = selective_curve(
    [score_level(a) == g for a, g in zip(test_answers, test_gold, strict=True)],
    [a.confidence for a in test_answers],
)
print("\nrisk-coverage curve (test), each distinct confidence value as a gate:")
for t, cov, acc, risk in zip(
    curve.thresholds, curve.coverage, curve.accuracy, curve.risk, strict=True
):
    print(f"  gate {t:.4f}: coverage {cov:.4f}, accuracy {acc:.4f}, risk {risk:.4f}{check}")

plot_risk_coverage(
    curve,
    reference_risk=1 - exact_agreement(test_gold, test_answers),
    title=f"Risk and coverage, test{check}",
)

## The retention decision itself

`classify_chunk` has a review branch beyond the confidence gate -- the retention cutoff -- so its own accuracy is reported from its actual outcomes (`jev_cookbook.evaluation.evaluate_outcomes`), not from re-deriving a confidence-only curve that cannot see the cutoff. "Accepted" means `classify_chunk` trusted the rating at all (outcome is not `review`); "correct" means the trusted candidate/paged-out call agrees with whether the gold level actually clears the retention cutoff (`SUPPORTING` or above).


In [ ]:
def retention_report(split_examples):
    classifications = [
        helpers.classify_chunk(e.fields["chunk_id"], decide(e), gate) for e in split_examples
    ]
    counts = {}
    for c in classifications:
        counts[c.outcome] = counts.get(c.outcome, 0) + 1
    accepted = [c.outcome != helpers.REVIEW for c in classifications]
    correct = [
        (c.outcome == helpers.CANDIDATE) == (labels[e.id] >= helpers.RETENTION_CUTOFF)
        for e, c in zip(split_examples, classifications, strict=True)
    ]
    return classifications, counts, evaluate_outcomes(accepted, correct)


for split_name, split_examples, label in (
    ("validation", val_examples, selection + check),
    ("test", test_examples, check),
):
    classifications, counts, outcome = retention_report(split_examples)
    print(f"{split_name}, {len(split_examples)} chunk ratings{label}")
    for name in (helpers.CANDIDATE, helpers.PAGED_OUT, helpers.REVIEW):
        print(f"  {name}: {counts.get(name, 0)} of {len(split_examples)}{label}")
    print(
        f"  retention call correct: {outcome.accuracy:.4f} "
        f"(risk {outcome.risk:.4f}) of {outcome.n_answered} trusted{label}"
    )
    reviewed = [
        e.id
        for e, c in zip(split_examples, classifications, strict=True)
        if c.outcome == helpers.REVIEW
    ]
    if reviewed:
        print(f"  e.g. sent to review: {reviewed[0]}{label}")

## A stateful case: paged out, then rehydrated

The three tasks above are independent: each scores its own chunk pool once. The use case also names stateful workflows, so this section walks through one ongoing investigation -- three turns over order ORD-8800, sharing one `jev_cookbook.simulation.Store` -- where the task changes turn to turn, a chunk already paged out gets reconsidered, and a brand-new chunk turns out to be a decoy rather than something to act on.

- **Turn 1**: the refund amount a customer was told doesn't match what was approved. The `NEEDED` chunk (the approved figure) is the *oldest* of four -- `allocate_budget` still finds it, where a recency-only baseline would not have.
- **Turn 2**: the task sharpens to *why* the two figures differ. A new chunk (the fee explanation) arrives and is immediately `NEEDED`; the two previously-`NEEDED`/`SUPPORTING` chunks both remain useful context and stay retained.
- **Turn 3**: a second, different customer reports the same pattern. A brand-new chunk -- an unreviewed support macro that contradicts the documented fee behaviour -- arrives newest and on-topic by its own words, and is still correctly rated `NOT_RELEVANT`. Meanwhile the general billing-microservice note, paged out on both earlier turns, becomes the single reusable explanation and is **rehydrated**.

`helpers.run_session` composes `classify_chunk`, `allocate_budget` and `apply_to_store` once per turn over the same `Store`, carrying each chunk's previous status forward so a later turn can rehydrate it.


In [ ]:
session_examples = [e for e in examples if e.fields["task_id"].startswith("s-t")]
by_turn = {}
for e in session_examples:
    by_turn.setdefault(e.fields["task_id"], []).append(e)

turns = []
for task_id in ("s-t1", "s-t2", "s-t3"):
    turn = [
        (
            e.fields["chunk_id"],
            {"content": e.fields["chunk"], "source": e.fields["source"]},
            decide(e),
        )
        for e in by_turn[task_id]
    ]
    turns.append(turn)

store, per_turn = helpers.run_session(turns, confidence_gate=gate, slots=helpers.BUDGET_SLOTS)
for task_id, (decisions, budget) in zip(("s-t1", "s-t2", "s-t3"), per_turn, strict=True):
    print(f"{task_id}: {by_turn[task_id][0].fields['task']}")
    print(f"  budget used: {budget.used('slots')} of {budget.limit('slots')}")
    for d in decisions:
        flag = " REHYDRATED" if d.rehydrated else ""
        print(f"  {d.chunk_id}: level {d.classification.level} -> {d.status}{flag}")

### Lossless, including the rehydration

A round trip that kept the live `store` object and compared it to itself would prove nothing about whether paging loses anything -- it would pass even if `apply_to_store` silently dropped every source link. The check below instead serialises the store's entire write history to JSON text, rebuilds a brand-new `Store` from that text with `helpers.rebuild_store`, and compares the rebuilt store to the original field by field, for the one chunk that was paged out twice and then rehydrated.


In [ ]:
import json

dumped = json.loads(json.dumps(store.to_dicts()))
rebuilt = helpers.rebuild_store(dumped)
print(f"rebuilt snapshot matches the original: {rebuilt.snapshot() == store.snapshot()}")

rehydrated_id = "s-c3"
original_history = [r.to_dict() for r in store.history(rehydrated_id)]
rebuilt_history = [r.to_dict() for r in rebuilt.history(rehydrated_id)]
print(f"\n{rehydrated_id} status across turns: {[r['value']['status'] for r in original_history]}")
print(
    f"{rehydrated_id} content identical on every write: "
    f"{helpers.content_preserved(store.history(rehydrated_id))}"
)
print(
    f"{rehydrated_id} source identical on every write: "
    f"{len({r['value']['source'] for r in original_history}) == 1}"
)
print(
    f"{rehydrated_id} history reconstructed from JSON, byte for byte: "
    f"{rebuilt_history == original_history}"
)

# Prove the round trip can actually detect a loss, rather than passing no matter what: tamper
# with one serialised write's content before rebuilding, and confirm the rebuilt store disagrees
# with the original on that key.
tampered = json.loads(json.dumps(store.to_dicts()))
for record in tampered:
    if record["key"] == rehydrated_id and record["step"] == 2:
        record["value"]["content"] = "TAMPERED"
tampered_store = helpers.rebuild_store(tampered)
print(
    f"\na tampered rebuild is detectably different: "
    f"{tampered_store.get(rehydrated_id)['content'] != store.get(rehydrated_id)['content']}"
)

## What was and was not measured

What this run can say about Jev depends on its mode, stated below from `backend.mode`: a synthetic or scripted run is a pipeline check only, on the fixture sample it ran against; a recorded or live run states the model the API returned and when, for exactly the examples it covers. Nothing above says anything about Jev's quality, speed or cost unless the cell below reports a recorded or live run. `N` counts every chunk rating with a gold label (35 test, 30 validation), which excludes the 4 demo chunks shown earlier.


In [ ]:
if offline:
    print(f"Provenance: {backend.mode}. Not measured live: a pipeline check, not a Jev result.")
    print(f"N: {len(scored)} scored examples{check}")
else:
    dates = ", ".join(getattr(backend, "recorded_dates", ()) or ()) or "this run"
    print(f"Provenance: {backend.mode}, model {backend.model}")
    print(f"Captured: {dates}")
    print(f"N: {len(scored)} scored examples")

## Next steps

- Change `helpers.BUDGET_SLOTS` from 3 to 2 and re-run the session cell: turn 1 can no longer retain both the approved-amount chunk and the email-amount chunk, and `allocate_budget`'s ranking (not list position) decides which one goes.
- Add a task whose pool has two `NEEDED` chunks instead of one, run `python build_fixtures.py`, and watch `recall_at_budget` report a fractional recall when only one of the two fits under the budget.
- Neighbouring recipes: [`17-passage-reranking`](../17-passage-reranking/), which this recipe builds on for the one-`Score`-per-candidate pattern and the lexical baseline this recipe's recency baseline plays the same role as; [`32-regression-test-selection`](../32-regression-test-selection/), another recipe that selects a bounded subset under a budget; and [`43-failure-recovery-selection`](../43-failure-recovery-selection/), another Level 4 recipe whose stateful rules are enforced the same way, with `jev_cookbook.simulation.Budget`.
